In [46]:
import pandas as pd
import os
from tqdm import tqdm
import json

DO_ALL_MEETINGS = True
AMI_PATH = '/home/sgoyal/Projects/llm_asr_clarification/shared/datasets/amicorpus/train'
# AMI_PATH = '/home/pkongsomjit/Projects/llm_asr_clarification/shared/datasets/amicorpus/train'
# AMI_PATH = '/home/surigo/Projects/WPI-Research-Projects/Automatic-Speech-Recognition/Code/llm_asr_clarification/datasets/amicorpus'
MEETING_TO_DO= '/home/pkongsomjit/Projects/llm_asr_clarification/datasets/amicorpus/ES2005d'
# QUESTION_FILE = 'parsed_gt'
QUESTION_FILE = 'parsed_diarized_gt'
TRANSCRIPT_FILES = [
    # 'whisper_tiny_diarized_transcript', 
    # 'whisper_tiny_diarized_transcript_random_clarify',
    # 'whisper_tiny_diarized_transcript_llm-orig-ctx_clarify',
    # 'whisper_tiny_diarized_transcript_llm-gt-ctx_clarify',
    'parsed_diarized_gt',
    # 'custom_transcript_gt_segments_gt_clarify',
    # 'custom_transcript_gt_segments_gt_clarify2',
    # 'custom_transcript_gt_segments_clarify_only_importance',
    # 'custom_transcript_gt_segments_all_gt_clarify3',
    # 'custom_transcript_gt_segments_all_lstm_clarify3',
    # 'baseline_prompting',
    # 'custom_transcript_gt_segments_random_clarify',
    # 'custom_transcript_gt_segments_rf_clarify',
    'custom_transcript_gt_segments',
    # 'custom_transcript_gt_segments_noise',

    # SAMPLE FILES
    # 'whisper_tiny_diarized_transcript_llm-orig-ctx_clarify_sample2',
    # 'whisper_tiny_diarized_transcript_llm-gt-ctx_clarify_sample2',
]
TRANSCRIPT_FILES = [f'score_using_{t}' for t in TRANSCRIPT_FILES]

# directories of meetings
if DO_ALL_MEETINGS:
    meeting_paths = [entry.path for entry in os.scandir(AMI_PATH)]

list_of_quiz_dicts = []
for meeting_path in tqdm(meeting_paths):
    meeting_name = meeting_path.split("/")[-1]
    if meeting_name == "EN2009d":
        continue

    # question_path = os.path.join(meeting_path, "quiz", f"quiz_from_{QUESTION_FILE}.json")
    # question_path = os.path.join(meeting_path, "quiz", "old_quiz.json")
    question_path = os.path.join(meeting_path, "quiz", "vllm_quiz.json")

    # chatgpt = OpenAIWrapper()
    
    # Read question
    try:
        with open(question_path, "r", encoding="utf-8") as f:
            quiz = f.read()

        quiz = json.loads(quiz)
        meeting_name = meeting_path.split("/")[-1]
        for q in quiz:
            q['meeting_name'] = meeting_name
        list_of_quiz_dicts += quiz
    except Exception as err:
        print(f"couldnt open file {question_path}")

df = pd.DataFrame(list_of_quiz_dicts)

100%|██████████| 136/136 [00:00<00:00, 612.21it/s]


In [47]:
df.head()

,question,correct_answer,answer_using_parsed_diarized_gt,score_using_parsed_diarized_gt,answer_using_custom_transcript_gt_segments,score_using_custom_transcript_gt_segments,meeting_name
0,What was the main reason given for flattening ...,To save two Euros.,The main reason given for flattening the remot...,1,The main reason given for flattening the remot...,1,ES2005d
1,Why did the team decide not to prioritize addi...,Because the team was trying to save money and ...,The team decided not to prioritize adding the ...,1,The team decided not to prioritize adding the ...,1,ES2005d
2,What feature was described as the main interac...,Speech recognition.,speech recognition,1,speech technology,1,ES2005d
3,Why was the tomato-based prototype included ev...,"Because there was some red material left over,...",The tomato-based prototype was included becaus...,1,The transcript does not provide a clear reason...,0,ES2005d
4,What was the purpose of the yellow element on ...,It was the slogan that needed to be incorporated.,The purpose of the yellow element on the first...,1,The yellow element on the first prototype repr...,1,ES2005d


In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1350 entries, 0 to 1349
Data columns (total 7 columns):
 #   Column                                      Non-Null Count  Dtype
---  ------                                      --------------  -----
 0   question                                    1350 non-null   str  
 1   correct_answer                              1350 non-null   str  
 2   answer_using_parsed_diarized_gt             1350 non-null   str  
 3   score_using_parsed_diarized_gt              1350 non-null   int64
 4   answer_using_custom_transcript_gt_segments  1350 non-null   str  
 5   score_using_custom_transcript_gt_segments   1350 non-null   int64
 6   meeting_name                                1350 non-null   str  
dtypes: int64(2), str(5)
memory usage: 74.0 KB


In [49]:
for transcript_file in TRANSCRIPT_FILES:
    df[transcript_file] = pd.to_numeric(df[transcript_file], errors="coerce")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1350 entries, 0 to 1349
Data columns (total 7 columns):
 #   Column                                      Non-Null Count  Dtype
---  ------                                      --------------  -----
 0   question                                    1350 non-null   str  
 1   correct_answer                              1350 non-null   str  
 2   answer_using_parsed_diarized_gt             1350 non-null   str  
 3   score_using_parsed_diarized_gt              1350 non-null   int64
 4   answer_using_custom_transcript_gt_segments  1350 non-null   str  
 5   score_using_custom_transcript_gt_segments   1350 non-null   int64
 6   meeting_name                                1350 non-null   str  
dtypes: int64(2), str(5)
memory usage: 74.0 KB


In [50]:
for transcript_file in TRANSCRIPT_FILES:
    print(transcript_file)
    print(df[transcript_file].mean())

score_using_parsed_diarized_gt
0.9488888888888889
score_using_custom_transcript_gt_segments
0.8311111111111111


In [17]:
from scipy.stats import f_oneway, ttest_rel

stat, pvalue = f_oneway(
    *[df[t] for t in TRANSCRIPT_FILES]
)

print(f"Stat: {stat}")
print(f"P Value: {pvalue}")

Stat: nan
P Value: nan


In [7]:
import numpy as np

stat, pvalue = ttest_rel(
    df["score_using_custom_transcript_gt_segments"],
    df["score_using_custom_transcript_gt_segments_all_gt_clarify3"],
    alternative="less"
)

print(f"Stat: {stat}")
print(f"P Value: {pvalue}")

Stat: -5.393128445322844
P Value: 4.07859416683111e-08


In [36]:
for transcript_file in TRANSCRIPT_FILES:
    print(transcript_file)
    print(df[transcript_file].value_counts())

score_using_parsed_diarized_gt
score_using_parsed_diarized_gt
1    1231
0     129
Name: count, dtype: int64
score_using_custom_transcript_gt_segments_gt_clarify
score_using_custom_transcript_gt_segments_gt_clarify
1    1086
0     274
Name: count, dtype: int64
score_using_custom_transcript_gt_segments_gt_clarify2
score_using_custom_transcript_gt_segments_gt_clarify2
1    1143
0     217
Name: count, dtype: int64
score_using_custom_transcript_gt_segments_clarify_only_importance
score_using_custom_transcript_gt_segments_clarify_only_importance
1    1177
0     183
Name: count, dtype: int64
score_using_custom_transcript_gt_segments
score_using_custom_transcript_gt_segments
1    1068
0     292
Name: count, dtype: int64


In [18]:
stats_dict = {}
for transcript_file in TRANSCRIPT_FILES:
    print(f"Grouped stats for {transcript_file}")
    stats = df.groupby("meeting_name")[transcript_file].describe()
    print(stats)
    stats_dict[transcript_file] = stats

Grouped stats for score_using_parsed_diarized_gt
              count  mean       std  min  25%  50%  75%  max
meeting_name                                                
EN2001a        10.0   1.0  0.000000  1.0  1.0  1.0  1.0  1.0
EN2001b        10.0   1.0  0.000000  1.0  1.0  1.0  1.0  1.0
EN2001d        10.0   1.0  0.000000  1.0  1.0  1.0  1.0  1.0
EN2001e        10.0   1.0  0.000000  1.0  1.0  1.0  1.0  1.0
EN2003a        10.0   1.0  0.000000  1.0  1.0  1.0  1.0  1.0
...             ...   ...       ...  ...  ...  ...  ...  ...
TS3011d        10.0   0.9  0.316228  0.0  1.0  1.0  1.0  1.0
TS3012a        10.0   0.9  0.316228  0.0  1.0  1.0  1.0  1.0
TS3012b        10.0   1.0  0.000000  1.0  1.0  1.0  1.0  1.0
TS3012c        10.0   0.9  0.316228  0.0  1.0  1.0  1.0  1.0
TS3012d        10.0   0.9  0.316228  0.0  1.0  1.0  1.0  1.0

[136 rows x 8 columns]


In [19]:
for transcript_file in TRANSCRIPT_FILES:
    print(f"Global stats by group for {transcript_file}")
    print(f"Mean across groups: {stats_dict[transcript_file]['mean'].mean()}")
    print(f"STD across groups: {stats_dict[transcript_file]['mean'].std()}")
    print(f"Min across groups: {stats_dict[transcript_file]['mean'].min()}")
    print(f"Max across groups: {stats_dict[transcript_file]['mean'].max()}")
    print()

Global stats by group for score_using_parsed_diarized_gt
Mean across groups: 0.9496296296296295
STD across groups: 0.07810638943618713
Min across groups: 0.5
Max across groups: 1.0

